# NullVector vectorless Q&A notebook

This notebook:

- builds or reuses NullVector acquisition artifacts
- builds or reuses a **deterministic** tree with `summarize=False` and `gateway=None`
- loads the canonical text substrate and committed `NodeCard` artifacts
- builds a **vectorless** retriever over NullVector-native artifacts
- layers an in-memory LangGraph Q&A flow on top
- uses LiteLLM through NullVector for answer generation

Before running, update:

- `SOURCE_PATH`
- `LITELLM_API_KEY`

The OpenRouter model and base URL are already wired directly in notebook code.


In [1]:
%pip install -U langgraph litellm

/home/pruthvi/projects/NullVector/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import json
import re
import string
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from pydantic import TypeAdapter
from typing_extensions import TypedDict

REPO_SRC = Path("/home/pruthvi/projects/NullVector/src").resolve()
if not REPO_SRC.exists():
    raise FileNotFoundError(f"NullVector src path does not exist: {REPO_SRC}")

if str(REPO_SRC) not in sys.path:
    sys.path.insert(0, str(REPO_SRC))

print("Using NullVector src:", REPO_SRC)

Using NullVector src: /home/pruthvi/projects/NullVector/src


In [3]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph

from nullvector.domain.common import NonEmptyStr, StrataModel
from nullvector.domain.models import (
    AcquisitionRequest,
    AcquisitionRunManifest,
    CanonicalTextPage,
    CanonicalTextSubstrate,
    ContentSpan,
    NodeCard,
    NodeSummary,
    TreeBuildManifest,
    TreeBuildRequest,
    VerificationReport,
)
from nullvector.ingest import AcquisitionService
from nullvector.llm import (
    GatewayAuditConfig,
    GatewayConfig,
    GatewayRequest,
    GatewayRetryPolicy,
    GatewayService,
    LiteLLMProviderConfig,
    LLMMessage,
    LLMRole,
)
from nullvector.observability import EventBus, JsonLoggerSubscriber
from nullvector.tree import TreePipelineService

In [4]:
SOURCE_PATH = "/home/pruthvi/projects/NullVector/cookbook/903000608.pdf"

ACQUISITION_RUN_ID = "acq-20260313-0001"
TREE_RUN_ID = "tree-20260313-0001"

ARTIFACT_ROOT = str((REPO_SRC.parent / "artifacts" / "acquisition_runs").resolve())

TOP_K_NODES = 5
TOP_K_LINES_PER_NODE = 4

# Direct LiteLLM / OpenRouter configuration
LITELLM_MODEL = "openrouter/google/gemini-3.1-flash-lite-preview"
LITELLM_API_BASE = "https://openrouter.ai/api/v1"

# Paste your OpenRouter API key here.
# Intentionally left as a placeholder so you do not accidentally persist a live secret.
LITELLM_API_KEY = "REDACTED_OPENROUTER_KEY"

if not LITELLM_API_KEY or LITELLM_API_KEY == "PASTE_YOUR_OPENROUTER_KEY_HERE":
    print("Warning: replace LITELLM_API_KEY before running Q&A cells")

print("ARTIFACT_ROOT =", ARTIFACT_ROOT)
print("LITELLM_MODEL =", LITELLM_MODEL)
print("LITELLM_API_BASE =", LITELLM_API_BASE)

ARTIFACT_ROOT = /home/pruthvi/projects/NullVector/artifacts/acquisition_runs
LITELLM_MODEL = openrouter/google/gemini-3.1-flash-lite-preview
LITELLM_API_BASE = https://openrouter.ai/api/v1


In [5]:
def _workflow_log_path(tree_run_id: str) -> Path:
    return Path(ARTIFACT_ROOT) / "_workflow_observability" / tree_run_id / "events.jsonl"


def _event_bus(tree_run_id: str) -> EventBus:
    return EventBus(subscribers=(JsonLoggerSubscriber(str(_workflow_log_path(tree_run_id))),))


def _qa_gateway() -> GatewayService:
    provider = LiteLLMProviderConfig(
        model=LITELLM_MODEL,
        api_key=LITELLM_API_KEY,
        api_base=LITELLM_API_BASE,
    )

    config = GatewayConfig(
        provider=provider,
        timeout_seconds=60.0,
        retry_policy=GatewayRetryPolicy(
            max_attempts=3,
            initial_backoff_seconds=0.5,
            backoff_multiplier=2.0,
            max_backoff_seconds=4.0,
        ),
        audit=GatewayAuditConfig(
            persist_root=str(Path(ARTIFACT_ROOT) / "_qa_gateway_audit"),
            capture_raw_request=True,
            capture_raw_response=True,
        ),
        structured_output_mode_preference=None,
    )
    return GatewayService(config)


def _read_text(path: str | Path) -> str:
    return Path(path).read_text(encoding="utf-8")


def _load_node_cards(path: str | Path) -> tuple[NodeCard, ...]:
    return TypeAdapter(tuple[NodeCard, ...]).validate_json(_read_text(path))


def _load_node_summaries(path: str | Path | None) -> tuple[NodeSummary, ...]:
    if path is None:
        return ()
    return TypeAdapter(tuple[NodeSummary, ...]).validate_json(_read_text(path))


def _load_tree_manifest(path: str | Path) -> TreeBuildManifest:
    return TreeBuildManifest.model_validate_json(_read_text(path))


def _load_acquisition_manifest(path: str | Path) -> AcquisitionRunManifest:
    return AcquisitionRunManifest.model_validate_json(_read_text(path))


def _load_canonical_text_substrate(path: str | Path) -> CanonicalTextSubstrate:
    return CanonicalTextSubstrate.model_validate_json(_read_text(path))


def _load_verification_report(path: str | Path) -> VerificationReport:
    return VerificationReport.model_validate_json(_read_text(path))

## Acquisition

This builds or reuses the NullVector acquisition artifacts.


In [6]:
acquisition_service = AcquisitionService(event_bus=_event_bus(TREE_RUN_ID))
acquisition_manifest = acquisition_service.acquire(
    AcquisitionRequest(
        source_path=SOURCE_PATH,
        acquisition_run_id=ACQUISITION_RUN_ID,
        artifact_root=ARTIFACT_ROOT,
    )
)

print("Acquisition artifact root:", acquisition_manifest.artifact_root)
print("Document ID:", acquisition_manifest.document_id)
print("Canonical text substrate path:", acquisition_manifest.canonical_text_substrate_path)

Acquisition artifact root: /home/pruthvi/projects/NullVector/artifacts/acquisition_runs/acq-20260313-0001/798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896
Document ID: 798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896
Canonical text substrate path: /home/pruthvi/projects/NullVector/artifacts/acquisition_runs/acq-20260313-0001/798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896/projection/canonical-text-substrate.json


## Deterministic tree build

This intentionally uses:

- `summarize=False`
- `gateway=None`

So the expected output is structural tree artifacts only, with no node summaries.


In [7]:
tree_service = TreePipelineService(event_bus=_event_bus(TREE_RUN_ID))
tree_manifest = tree_service.build(
    TreeBuildRequest(
        acquisition_manifest_path=str(Path(acquisition_manifest.artifact_root) / "manifest.json"),
        tree_run_id=TREE_RUN_ID,
        summarize=False,
    ),
    gateway=None,
)

print("Tree artifact root:", tree_manifest.artifact_root)
print("Node cards path:", tree_manifest.node_cards_path)
print("Node summaries path:", tree_manifest.node_summaries_path)

Tree artifact root: /home/pruthvi/projects/NullVector/artifacts/acquisition_runs/acq-20260313-0001/798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896/tree/tree-20260313-0001
Node cards path: /home/pruthvi/projects/NullVector/artifacts/acquisition_runs/acq-20260313-0001/798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896/tree/tree-20260313-0001/strategy/attempts/01-inferred_with_llm_assist/hierarchy/node-cards.json
Node summaries path: None


In [8]:
print("tree_manifest.artifact_root =", tree_manifest.artifact_root)
print("tree_manifest.node_cards_path =", tree_manifest.node_cards_path)
print("tree_manifest.node_summaries_path =", tree_manifest.node_summaries_path)
print("tree_manifest.verification_report_path =", tree_manifest.verification_report_path)
print("tree_manifest.build_report_path =", tree_manifest.build_report_path)
print(
    "tree_manifest.strategy_execution_report_path =", tree_manifest.strategy_execution_report_path
)
print("tree_manifest.decomposition_report_path =", tree_manifest.decomposition_report_path)

tree_manifest.artifact_root = /home/pruthvi/projects/NullVector/artifacts/acquisition_runs/acq-20260313-0001/798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896/tree/tree-20260313-0001
tree_manifest.node_cards_path = /home/pruthvi/projects/NullVector/artifacts/acquisition_runs/acq-20260313-0001/798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896/tree/tree-20260313-0001/strategy/attempts/01-inferred_with_llm_assist/hierarchy/node-cards.json
tree_manifest.node_summaries_path = None
tree_manifest.verification_report_path = /home/pruthvi/projects/NullVector/artifacts/acquisition_runs/acq-20260313-0001/798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896/tree/tree-20260313-0001/strategy/attempts/01-inferred_with_llm_assist/verify/report.json
tree_manifest.build_report_path = /home/pruthvi/projects/NullVector/artifacts/acquisition_runs/acq-20260313-0001/798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896/tree/tree-20260313-0001/strategy/atte

In [9]:
assert tree_manifest.node_summaries_path is None, (
    "summarize=False should produce no node summaries artifact"
)

node_cards = _load_node_cards(tree_manifest.node_cards_path)
verification_report = _load_verification_report(tree_manifest.verification_report_path)
acquisition_manifest_from_tree = _load_acquisition_manifest(tree_manifest.acquisition_manifest_path)

assert acquisition_manifest_from_tree.canonical_text_substrate_path is not None
text_substrate = _load_canonical_text_substrate(
    acquisition_manifest_from_tree.canonical_text_substrate_path
)

print("Loaded node cards:", len(node_cards))
print("Verification status:", verification_report.status)
print("Text substrate pages:", len(text_substrate.pages))

Loaded node cards: 3000
Verification status: passed
Text substrate pages: 148


In [10]:
for idx, node in enumerate(node_cards[:5], start=1):
    print(
        {
            "rank": idx,
            "node_id": node.node_id,
            "title": node.title,
            "path": node.path,
            "level": node.level,
            "page_span": (node.page_span.start_page, node.page_span.end_page),
            "summary": node.summary,
            "keywords": node.keywords,
            "summary_method": node.summary_method,
            "summary_token_count": node.summary_token_count,
        }
    )

{'rank': 1, 'node_id': 'b72cef9053fc86e4be077af1875741112039b775868e77c585c6ed8efdc9fcd1', 'title': 'The Constitution of India', 'path': ('The Constitution of India',), 'level': 1, 'page_span': (1, 11), 'summary': None, 'keywords': (), 'summary_method': None, 'summary_token_count': None}
{'rank': 2, 'node_id': 'a64e79575e742e927a25cd80eab173bce22c936e163bd4c44f58f60e0b4137b4', 'title': 'Chapter IV A', 'path': ('The Constitution of India', 'Chapter IV A'), 'level': 2, 'page_span': (1, 1), 'summary': None, 'keywords': (), 'summary_method': None, 'summary_token_count': None}
{'rank': 3, 'node_id': '0b3fce0ba956b638330d7d89234ba39f236c2bbfb286cf907e9e5e5e4563f484', 'title': 'Fundamental Duties', 'path': ('The Constitution of India', 'Fundamental Duties'), 'level': 2, 'page_span': (1, 2), 'summary': None, 'keywords': (), 'summary_method': None, 'summary_token_count': None}
{'rank': 4, 'node_id': 'a1147d47018b372a903d9586f5261aa50e6a079e2b4551a3c5f6259309b0f594', 'title': 'Sachin Mehta', 'pa

In [11]:
for node in node_cards:
    assert node.summary is None, f"Expected summary=None for node {node.node_id}"
    assert tuple(node.keywords) == (), f"Expected empty keywords for node {node.node_id}"
    assert node.summary_method is None, f"Expected summary_method=None for node {node.node_id}"
    assert node.summary_token_count is None, (
        f"Expected summary_token_count=None for node {node.node_id}"
    )

print("All node cards are structural-only, as expected for summarize=False")

All node cards are structural-only, as expected for summarize=False


## Canonical text extraction helpers


In [12]:
page_by_index: dict[int, CanonicalTextPage] = {
    page.page_index: page for page in text_substrate.pages
}


def _safe_slice(text: str, start: int, end: int) -> str:
    lo = max(0, min(start, len(text)))
    hi = max(lo, min(end, len(text)))
    return text[lo:hi]


def _extract_span_text(span: ContentSpan) -> str:
    parts: list[str] = []

    for page_index in range(span.start_page, span.end_page + 1):
        page = page_by_index.get(page_index)
        if page is None:
            continue

        start_offset = 0
        end_offset = len(page.text)

        if page_index == span.start_page:
            start_offset = span.start_offset
        if page_index == span.end_page:
            end_offset = span.end_offset

        piece = _safe_slice(page.text, start_offset, end_offset).strip()
        if piece:
            parts.append(piece)

    return "\n".join(parts).strip()


def extract_node_text(node: NodeCard) -> str:
    if node.owned_spans:
        chunks = [_extract_span_text(owned.span) for owned in node.owned_spans]
        text = "\n".join(chunk for chunk in chunks if chunk).strip()
        if text:
            return text

    fallback: list[str] = []
    for page_index in range(node.page_span.start_page, node.page_span.end_page + 1):
        page = page_by_index.get(page_index)
        if page is not None and page.text.strip():
            fallback.append(page.text.strip())
    return "\n".join(fallback).strip()

In [13]:
for idx, node in enumerate(node_cards[:3], start=1):
    node_text = extract_node_text(node)
    print(f"[Node {idx}] {node.title}")
    print(node_text[:3000])
    print("=" * 120)

[Node 1] The Constitution of India
The Constitution of India
Chapter IV A
Fundamental Duties
ARTICLE 51A
Fundamental Duties- It shall be the duty of every citizen of India-
(a)
to abide by the Constitution and respect its ideals and institutions,
the National Flag and the National Anthem;
(b)
to cherish and follow the noble ideals which inspired our national
struggle for freedom;
(c)
to uphold and protect the sovereignty, unity and integrity of India;
(d)
to defend the country and render national service when called upon
to do so;
(e)
to promote harmony and the spirit of common brotherhood amongst
all the people of India transcending religious, linguistic and regional
or sectional diversities, to renounce practices derogatory to the
dignity of women;
(f)
to value and preserve the rich heritage of our composite culture;
(g)
to protect and improve the natural environment including forests,
lakes, rivers and wild life and to have compassion for living
creatures;
(h)
to develop the scienti

## Vectorless retrieval


In [14]:
_PUNCT_TABLE = str.maketrans("", "", string.punctuation)
_WS_RE = re.compile(r"\s+")


def _normalize_text(value: str) -> str:
    return _WS_RE.sub(" ", value).strip().casefold()


def _tokens(value: str) -> tuple[str, ...]:
    normalized = _normalize_text(value).translate(_PUNCT_TABLE)
    return tuple(token for token in normalized.split() if token)


@dataclass(frozen=True)
class NodeRecord:
    node: NodeCard
    body_text: str
    title_text: str
    path_text: str
    search_text: str


def build_node_records(nodes: tuple[NodeCard, ...]) -> tuple[NodeRecord, ...]:
    records: list[NodeRecord] = []
    for node in nodes:
        body_text = extract_node_text(node)
        path_text = " > ".join(node.path)
        search_text = "\n".join(part for part in (node.title, path_text, body_text) if part).strip()
        records.append(
            NodeRecord(
                node=node,
                body_text=body_text,
                title_text=node.title,
                path_text=path_text,
                search_text=search_text,
            )
        )
    return tuple(records)


node_records = build_node_records(node_cards)
print("Node records built:", len(node_records))

Node records built: 3000


In [15]:
def _unique_overlap(query_tokens: tuple[str, ...], text_tokens: tuple[str, ...]) -> int:
    return len(set(query_tokens) & set(text_tokens))


def _score_record(question: str, record: NodeRecord) -> float:
    q_norm = _normalize_text(question)
    q_tokens = _tokens(question)

    title_tokens = _tokens(record.title_text)
    path_tokens = _tokens(record.path_text)
    body_tokens = _tokens(record.body_text)

    score = 0.0

    if q_norm and q_norm in _normalize_text(record.title_text):
        score += 40.0
    if q_norm and q_norm in _normalize_text(record.path_text):
        score += 25.0

    score += 8.0 * _unique_overlap(q_tokens, title_tokens)
    score += 5.0 * _unique_overlap(q_tokens, path_tokens)
    score += 1.5 * _unique_overlap(q_tokens, body_tokens)

    return score


def _top_support_lines(
    record: NodeRecord, question: str, top_k: int = TOP_K_LINES_PER_NODE
) -> list[dict[str, Any]]:
    q_tokens = _tokens(question)
    candidates: list[tuple[float, dict[str, Any]]] = []
    seen: set[tuple[int, str]] = set()

    for anchor in record.node.source_anchors:
        quote = anchor.quote.strip()
        if not quote:
            continue
        key = (anchor.page, quote)
        if key in seen:
            continue
        seen.add(key)
        candidates.append(
            (
                20.0 + 4.0 * _unique_overlap(q_tokens, _tokens(quote)),
                {
                    "page": anchor.page,
                    "start_offset": anchor.start_offset,
                    "end_offset": anchor.end_offset,
                    "quote": quote,
                    "kind": "source_anchor",
                },
            )
        )

    for page_index in range(record.node.page_span.start_page, record.node.page_span.end_page + 1):
        page = page_by_index.get(page_index)
        if page is None:
            continue
        for line in page.lines:
            overlap = _unique_overlap(q_tokens, _tokens(line.content))
            if overlap == 0:
                continue
            key = (line.page_index, line.content)
            if key in seen:
                continue
            seen.add(key)
            candidates.append(
                (
                    5.0 * overlap,
                    {
                        "page": line.page_index,
                        "start_offset": line.start_offset,
                        "end_offset": line.end_offset,
                        "quote": line.content,
                        "kind": "canonical_line",
                    },
                )
            )

    candidates.sort(key=lambda item: (-item[0], item[1]["page"], item[1]["start_offset"]))
    return [item[1] for item in candidates[:top_k]]


def retrieve_vectorless(
    question: str, top_k: int = TOP_K_NODES
) -> tuple[list[dict[str, Any]], str]:
    ranked: list[tuple[float, NodeRecord]] = []

    for record in node_records:
        score = _score_record(question, record)
        if score > 0:
            ranked.append((score, record))

    ranked.sort(
        key=lambda item: (
            -item[0],
            item[1].node.page_span.start_page,
            item[1].node.level,
            item[1].node.node_id,
        )
    )

    selected = ranked[:top_k]
    sources: list[dict[str, Any]] = []
    context_blocks: list[str] = []

    for rank, (score, record) in enumerate(selected, start=1):
        support_lines = _top_support_lines(record, question)

        payload = {
            "rank": rank,
            "score": round(score, 3),
            "node_id": record.node.node_id,
            "title": record.node.title,
            "path": list(record.node.path),
            "level": record.node.level,
            "page_span": {
                "start_page": record.node.page_span.start_page,
                "end_page": record.node.page_span.end_page,
            },
            "support_lines": support_lines,
            "excerpt": record.body_text[:2500],
        }
        sources.append(payload)

        support_text = (
            "\n".join(
                f"- page={line['page']} offsets=({line['start_offset']},{line['end_offset']}) quote={line['quote']!r}"
                for line in support_lines
            )
            or "- no support"
        )

        context_blocks.append(
            "\n".join(
                [
                    f"[Source {rank}]",
                    f"node_id={record.node.node_id}",
                    f"title={record.node.title}",
                    f"path={' > '.join(record.node.path)}",
                    f"level={record.node.level}",
                    f"page_span=({record.node.page_span.start_page}, {record.node.page_span.end_page})",
                    "support:",
                    support_text,
                    "excerpt:",
                    record.body_text[:2500],
                ]
            )
        )

    if not context_blocks:
        return [], "No supporting sources were retrieved."

    return sources, "\n\n".join(context_blocks)

In [16]:
test_question = "What are the main sections in this document?"
retrieved_sources, retrieved_context = retrieve_vectorless(test_question, top_k=TOP_K_NODES)

print("Retrieved source count:", len(retrieved_sources))
print(retrieved_context[:8000])

Retrieved source count: 5
[Source 1]
node_id=b815aa3f3bffbb4281dcabc852830ce76d774fe5f12e8b5a16857c9a36a76e1e
title=2 , 3 , 5 , .... these type of surds are in the simplest form which cannot be simplified
path=2 , 3 , 5 , .... these type of surds are in the simplest form which cannot be simplified
level=1
page_span=(35, 35)
support:
- page=35 offsets=(1300,1388) quote='2 , 3 , 5 , .... these type of surds are in the simplest form which cannot be simplified'
- page=35 offsets=(0,5) quote='Surds'
- page=35 offsets=(1082,1146) quote='This year we are going to study surds of order 2 only, means 3 ,'
- page=35 offsets=(498,508) quote='the symbol'
excerpt:
2 , 3 , 5 , .... these type of surds are in the simplest form which cannot be simplified
further.
Similar or like surds
2 , 4
2 , -3
2  are some like surds.
5
a ,  q
If p and q are rational numbers then p
a  are called like surds. Two surds are said
to be like surds if their order is equal and radicands are equal.
26

[Source 2]
node_id=8e

## Q&A schema and graph


In [19]:
class AnswerCitation(StrataModel):
    node_id: NonEmptyStr
    page_start: int
    page_end: int
    quote: NonEmptyStr


class QAAnswer(StrataModel):
    answer: NonEmptyStr
    insufficient_context: bool = False
    citations: tuple[AnswerCitation, ...] = ()


class QAState(TypedDict, total=False):
    question: str
    retrieved_sources: list[dict[str, Any]]
    retrieved_context: str
    answer: str
    insufficient_context: bool
    citations: list[dict[str, Any]]
    messages: list[dict[str, str]]

In [20]:
qa_gateway = _qa_gateway()


def retrieve_node(state: QAState) -> dict[str, Any]:
    sources, context = retrieve_vectorless(state["question"], top_k=TOP_K_NODES)
    return {
        "retrieved_sources": sources,
        "retrieved_context": context,
    }


def answer_node(state: QAState) -> dict[str, Any]:
    history = state.get("messages", [])
    history_text = (
        "\n".join(f"{item['role'].upper()}: {item['content']}" for item in history[-8:]) or "None"
    )

    messages = (
        LLMMessage(
            role=LLMRole.SYSTEM,
            content=(
                "You are a retrieval-grounded Q&A assistant over NullVector outputs. "
                "Use only the provided retrieved context. "
                "Do not invent facts. "
                "If the question is not answerable from the context, set insufficient_context=true "
                "and explain the gap. "
                "Every citation must correspond to a provided source and quote."
            ),
        ),
        LLMMessage(
            role=LLMRole.USER,
            content=(
                f"Conversation history:\n{history_text}\n\n"
                f"Retrieved context:\n{state['retrieved_context']}\n\n"
                f"Question:\n{state['question']}\n\n"
                "Return a structured answer."
            ),
        ),
    )

    response = qa_gateway.invoke(
        GatewayRequest[QAAnswer](
            operation_name="vectorless_document_qa",
            messages=messages,
            response_model=QAAnswer,
            temperature=0.0,
            metadata={"workflow": "nullvector_vectorless_qa"},
        )
    ).output

    updated_messages = [
        *history,
        {"role": "user", "content": state["question"]},
        {"role": "assistant", "content": response.answer},
    ]

    return {
        "answer": response.answer,
        "insufficient_context": response.insufficient_context,
        "citations": [citation.model_dump(mode="json") for citation in response.citations],
        "messages": updated_messages,
    }

In [21]:
def build_qa_graph(*, checkpointer: Any):
    builder = StateGraph(QAState)

    builder.add_node("retrieve", retrieve_node)
    builder.add_node("answer", answer_node)

    builder.add_edge(START, "retrieve")
    builder.add_edge("retrieve", "answer")
    builder.add_edge("answer", END)

    return builder.compile(checkpointer=checkpointer)


checkpointer = InMemorySaver()
qa_graph = build_qa_graph(checkpointer=checkpointer)

print("Compiled QA graph with InMemorySaver")

Compiled QA graph with InMemorySaver


## Ask questions


In [22]:
THREAD_ID = "sf-qa-thread-0001"
config = {"configurable": {"thread_id": THREAD_ID}}

question_1 = "What are the main sections in this document, and what does each cover?"

result_1 = qa_graph.invoke(
    {"question": question_1},
    config=config,
)

print(result_1["answer"])

The provided document covers several distinct mathematical and statistical topics across its sections, including surds, set theory, continued proportions, data visualization, and frequency distribution.



Provider List: https://docs.litellm.ai/docs/providers



In [23]:
print("Citations:")
print(json.dumps(result_1.get("citations", []), indent=2, default=str))

print("\nRetrieved sources:")
for source in result_1["retrieved_sources"]:
    print(json.dumps(source, indent=2, default=str)[:2500])
    print("-" * 100)

Citations:
[
  {
    "node_id": "b815aa3f3bffbb4281dcabc852830ce76d774fe5f12e8b5a16857c9a36a76e1e",
    "page_start": 35,
    "page_end": 35,
    "quote": "2 , 3 , 5 , .... these type of surds are in the simplest form which cannot be simplified"
  },
  {
    "node_id": "4d149704f80c7007eda35cc785a822965418e16da5dba553f5af1467f3fb7ddc",
    "page_start": 12,
    "page_end": 12,
    "quote": "p  such that  p and q are integers"
  },
  {
    "node_id": "7d7d2dd79b7911030f52004eff70992713ae12d8d05c98253de2bc773cb47015",
    "page_start": 85,
    "page_end": 85,
    "quote": "Ex  (4) Five numbers are in continued"
  },
  {
    "node_id": "3f89c513feed4b6f4c18062db7a876f45104696d3b9a10c06eaf0fc484816a9d",
    "page_start": 119,
    "page_end": 119,
    "quote": "The information is shown in the percentage bar diagram by following steps"
  },
  {
    "node_id": "6def96829445a33f187cc42dcee52d7bb10c6119cde3c30f2da7c7eeaf1b8ed0",
    "page_start": 125,
    "page_end": 125,
    "quote": "Basic te

In [24]:
question_2 = (
    "Which section seems most important for implementation, and what evidence supports that?"
)

result_2 = qa_graph.invoke(
    {"question": question_2},
    config=config,
)

print(result_2["answer"])
print(json.dumps(result_2.get("citations", []), indent=2, default=str))


Provider List: https://docs.litellm.ai/docs/providers

The section on 'Savings and Investments' appears to be the most important for practical implementation, as it addresses the necessity of financial planning for both predictable and unpredictable life events. The document emphasizes that financial planning is a 'must' because it serves the purpose of protecting and growing wealth to meet future needs, such as education, business capital, and emergency medical expenses.
[
  {
    "node_id": "7b78a462027c9f7465973bfe6253107b137dc192426a240896c27b69df6dd60c",
    "page_start": 102,
    "page_end": 102,
    "quote": "The above considerations make it quite clear why financial planning is a must. However some important points must be kept in mind as we plan our finances."
  },
  {
    "node_id": "7b78a462027c9f7465973bfe6253107b137dc192426a240896c27b69df6dd60c",
    "page_start": 102,
    "page_end": 102,
    "quote": "The main purpose of financial planning is protection and growth of th

In [25]:
snapshot = qa_graph.get_state(config)
print(json.dumps(snapshot.values, indent=2, sort_keys=True, default=str)[:10000])

{
  "answer": "The section on 'Savings and Investments' appears to be the most important for practical implementation, as it addresses the necessity of financial planning for both predictable and unpredictable life events. The document emphasizes that financial planning is a 'must' because it serves the purpose of protecting and growing wealth to meet future needs, such as education, business capital, and emergency medical expenses.",
  "citations": [
    {
      "node_id": "7b78a462027c9f7465973bfe6253107b137dc192426a240896c27b69df6dd60c",
      "page_end": 102,
      "page_start": 102,
      "quote": "The above considerations make it quite clear why financial planning is a must. However some important points must be kept in mind as we plan our finances."
    },
    {
      "node_id": "7b78a462027c9f7465973bfe6253107b137dc192426a240896c27b69df6dd60c",
      "page_end": 102,
      "page_start": 102,
      "quote": "The main purpose of financial planning is protection and growth of the 

In [26]:
question_3 = "what is the image on first page about?"

result_3 = qa_graph.invoke(
    {"question": question_3},
    config=config,
)

print(result_3["answer"])
print(json.dumps(result_3.get("citations", []), indent=2, default=str))


Provider List: https://docs.litellm.ai/docs/providers

The provided context does not contain information about the first page of the document, as the retrieved sources only include pages 16, 30, 44, 83, 92, and 119.
[]


In [27]:
qa_gateway.close()
print("Closed QA gateway")

Closed QA gateway


In [28]:
covered_by_nodes = [
    {
        "node_id": node.node_id,
        "title": node.title,
        "page_span": (node.page_span.start_page, node.page_span.end_page),
    }
    for node in node_cards
    if node.page_span.start_page <= 0 <= node.page_span.end_page
]

covered_by_nodes[:10], len(covered_by_nodes)

([], 0)

In [29]:
[
    {
        "reason": span.reason,
        "page_span": (span.page_span.start_page, span.page_span.end_page),
    }
    for span in verification_report.unassigned_spans
    if span.page_span.start_page <= 0 <= span.page_span.end_page
]

[{'reason': 'before_first_heading', 'page_span': (0, 0)}]

In [30]:
first_page = next(page for page in text_substrate.pages if page.page_index == 0)
print("page_index:", first_page.page_index)
print("text_len:", len(first_page.text))
print(first_page.text[:2000])

page_index: 0
text_len: 0



In [31]:
from pathlib import Path

from nullvector.domain.models import CanonicalDocumentLedger

ledger_path = Path(acquisition_manifest.ledger_path)
ledger = CanonicalDocumentLedger.model_validate_json(ledger_path.read_text(encoding="utf-8"))

page0 = next(page for page in ledger.pages if page.page_index == 0)

print("page0 block count:", len(page0.blocks))
for block in page0.blocks:
    print(type(block).__name__, getattr(block, "block_type", None))

page0 block count: 8
VisualArtifact visual_artifact
UnresolvedRegion unresolved_region
VisualArtifact visual_artifact
UnresolvedRegion unresolved_region
VisualArtifact visual_artifact
UnresolvedRegion unresolved_region
VisualArtifact visual_artifact
UnresolvedRegion unresolved_region


In [32]:
page0

CanonicalPage(page_index=0, page_label='', width=585.56396484375, height=809.4010009765625, rotation=0.0, native_available=False, blocks=(VisualArtifact(block_type='visual_artifact', visual_id='page-0-visual-0000', bbox=BoundingBox(x0=-31.6976318359375, y0=-5.30767822265625, x1=604.0637817382812, y1=816.6939086914062), reading_index=0, kind_hint='embedded_image', image_ref='page-0-image-0000', asset_path='/home/pruthvi/projects/NullVector/artifacts/acquisition_runs/acq-20260313-0001/798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896/assets/pages/000000/page-0-visual-0000.png', page_render_path='/home/pruthvi/projects/NullVector/artifacts/acquisition_runs/acq-20260313-0001/798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896/assets/pages/000000/render-144dpi.png', render_dpi=144, coordinate_space=<GeometryCoordinateSpace.UNROTATED_PAGE: 'unrotated_page'>, needs_enrichment=True, provenance=ExtractionProvenance(source_track=<SourceTrack.NATIVE: 'native'>, produce

In [34]:
# Visual QA imports + ledger load

import base64
import mimetypes
import re
from pathlib import Path

from litellm import completion

from nullvector.domain.models import CanonicalDocumentLedger, UnresolvedRegion, VisualArtifact

# assert supports_vision(model=LITELLM_MODEL), f"Model does not report vision support: {LITELLM_MODEL}"

ledger = CanonicalDocumentLedger.model_validate_json(
    Path(acquisition_manifest.ledger_path).read_text(encoding="utf-8")
)
ledger_pages = {page.page_index: page for page in ledger.pages}

print("Loaded ledger pages:", len(ledger_pages))
print("Vision supported for model:", LITELLM_MODEL)

Loaded ledger pages: 148
Vision supported for model: openrouter/google/gemini-3.1-flash-lite-preview


In [35]:
# Page / region selection helpers


def _bbox_area(block: Any) -> float:
    width = max(0.0, float(block.bbox.x1 - block.bbox.x0))
    height = max(0.0, float(block.bbox.y1 - block.bbox.y0))
    return width * height


def _select_visual_candidate_for_page(page_index: int) -> dict[str, Any] | None:
    page = ledger_pages.get(page_index)
    if page is None:
        return None

    candidates: list[dict[str, Any]] = []

    for block in page.blocks:
        if isinstance(block, UnresolvedRegion):
            asset_path = block.asset_path or block.page_render_path
            if asset_path:
                candidates.append(
                    {
                        "kind": "unresolved_region",
                        "page_index": page_index,
                        "region_id": block.region_id,
                        "reason_code": block.reason_code,
                        "asset_path": asset_path,
                        "page_render_path": block.page_render_path,
                        "recommended_fallback": block.recommended_fallback,
                        "area": _bbox_area(block),
                    }
                )
        elif isinstance(block, VisualArtifact):
            asset_path = block.asset_path or block.page_render_path
            if asset_path:
                candidates.append(
                    {
                        "kind": "visual_artifact",
                        "page_index": page_index,
                        "region_id": block.visual_id,
                        "reason_code": None,
                        "asset_path": asset_path,
                        "page_render_path": block.page_render_path,
                        "recommended_fallback": "visual_enrichment",
                        "area": _bbox_area(block),
                    }
                )

    if not candidates:
        return None

    # Prefer the largest region; if tied, prefer unresolved_region because it explicitly
    # encodes the framework's fallback contract.
    candidates.sort(
        key=lambda item: (
            -item["area"],
            0 if item["kind"] == "unresolved_region" else 1,
            item["region_id"],
        )
    )
    return candidates[0]


page0_visual = _select_visual_candidate_for_page(0)
page0_visual

{'kind': 'unresolved_region',
 'page_index': 0,
 'region_id': 'page-0-unresolved-0000',
 'reason_code': 'image_only_region',
 'asset_path': '/home/pruthvi/projects/NullVector/artifacts/acquisition_runs/acq-20260313-0001/798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896/assets/pages/000000/page-0-unresolved-0000.png',
 'page_render_path': '/home/pruthvi/projects/NullVector/artifacts/acquisition_runs/acq-20260313-0001/798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896/assets/pages/000000/render-144dpi.png',
 'recommended_fallback': 'external_ocr_or_visual_enrichment',
 'area': 522596.8908567354}

In [36]:
# Visual-question routing helpers

_ORDINAL_PAGE_WORDS = {
    "first": 0,
    "second": 1,
    "third": 2,
    "fourth": 3,
    "fifth": 4,
    "sixth": 5,
    "seventh": 6,
    "eighth": 7,
    "ninth": 8,
    "tenth": 9,
}


def is_visual_question(question: str) -> bool:
    q = question.casefold()
    visual_terms = (
        "image",
        "figure",
        "diagram",
        "chart",
        "photo",
        "picture",
        "illustration",
        "logo",
        "cover",
        "first page",
        "page 1 image",
    )
    return any(term in q for term in visual_terms)


def infer_requested_page_index(question: str) -> int | None:
    q = question.casefold().strip()

    if "cover page" in q or "front page" in q or "first page" in q:
        return 0

    for word, index in _ORDINAL_PAGE_WORDS.items():
        if f"{word} page" in q:
            return index
        if f"page {word}" in q:
            return index

    ordinal_match = re.search(r"\b(\d+)(?:st|nd|rd|th)\s+page\b", q)
    if ordinal_match:
        return max(0, int(ordinal_match.group(1)) - 1)

    page_match = re.search(r"\bpage\s+(\d+)\b", q)
    if page_match:
        # User-facing page numbers are treated as 1-based.
        return max(0, int(page_match.group(1)) - 1)

    return None


print(is_visual_question("what is the image on first page about?"))
print(infer_requested_page_index("what is the image on first page about?"))

True
0


In [37]:
# Image encoding + direct LiteLLM visual call


def _data_url_for_image(path: str | Path) -> str:
    path = Path(path)
    mime_type = mimetypes.guess_type(path.name)[0] or "image/png"
    payload = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:{mime_type};base64,{payload}"


def _message_content_to_text(content: Any) -> str:
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        chunks: list[str] = []
        for item in content:
            if isinstance(item, str):
                chunks.append(item)
            elif isinstance(item, dict):
                text = item.get("text")
                if isinstance(text, str):
                    chunks.append(text)
        return "\n".join(chunk for chunk in chunks if chunk).strip()
    return str(content)


def answer_visual_question_direct(
    *,
    question: str,
    page_index: int,
) -> dict[str, Any]:
    selected = _select_visual_candidate_for_page(page_index)
    if selected is None:
        return {
            "answer": f"No visual asset could be resolved for page {page_index}.",
            "page_index": page_index,
            "selected_asset": None,
            "citation_quote": f"visual page analysis unavailable for page {page_index}",
        }

    image_path = selected["asset_path"]
    data_url = _data_url_for_image(image_path)
    mime_type = mimetypes.guess_type(str(image_path))[0] or "image/png"

    response = completion(
        model=LITELLM_MODEL,
        api_key=LITELLM_API_KEY,
        api_base=LITELLM_API_BASE,
        temperature=0.0,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": (
                            "You are answering a question about a document image.\n"
                            "Use only the provided image.\n"
                            "Do not invent details.\n"
                            "If the image is ambiguous, say so.\n\n"
                            f"Question: {question}\n\n"
                            "Return a concise answer in 2-5 sentences."
                        ),
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": data_url,
                            "format": mime_type,
                        },
                    },
                ],
            }
        ],
    )

    answer_text = _message_content_to_text(response.choices[0].message.content).strip()

    return {
        "answer": answer_text,
        "page_index": page_index,
        "selected_asset": selected,
        "citation_quote": (f"visual analysis of page {page_index} using {Path(image_path).name}"),
    }


visual_debug = answer_visual_question_direct(
    question="what is the image on first page about?",
    page_index=0,
)

print(visual_debug["answer"])
print(visual_debug["selected_asset"])


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers

The image is the cover of a "Mathematics Part-I" textbook for "Standard Nine." It depicts a girl and a boy studying together at a desk, with the boy using a compass to draw in a notebook. A chalkboard in the background displays algebraic formulas.
{'kind': 'unresolved_region', 'page_index': 0, 'region_id': 'page-0-unresolved-0000', 'reason_code': 'image_only_region', 'asset_path': '/home/pruthvi/projects/NullVector/artifacts/acquisition_runs/acq-20260313-0001/798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896/assets/pages/000000/page-0-unresolved-0000.png', 'page_render_path': '/home/pruthvi/projects/NullVector/artifacts/acquisition_runs/acq-20260313-0001/798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896/assets/pages/000000/render-144dpi.png', 'recommended_fallback': 'external_ocr_or_visual_enrichment', 'area': 522596.8908567354}

Provider List: h

In [38]:
# New routed graph: visual questions go to direct visual analysis, everything else stays on your existing text path

from langgraph.checkpoint.memory import InMemorySaver
from typing_extensions import TypedDict


class RoutedQAState(TypedDict, total=False):
    question: str
    route_kind: str
    visual_page_index: int
    retrieved_sources: list[dict[str, Any]]
    retrieved_context: str
    answer: str
    insufficient_context: bool
    citations: list[dict[str, Any]]
    messages: list[dict[str, str]]


def route_question_node(state: RoutedQAState) -> dict[str, Any]:
    question = state["question"]

    if is_visual_question(question):
        page_index = infer_requested_page_index(question)
        if page_index is not None and _select_visual_candidate_for_page(page_index) is not None:
            return {
                "route_kind": "visual",
                "visual_page_index": page_index,
            }

    return {
        "route_kind": "text",
    }


def route_question_edge(state: RoutedQAState) -> str:
    return "visual_answer" if state.get("route_kind") == "visual" else "retrieve"


def visual_answer_node(state: RoutedQAState) -> dict[str, Any]:
    question = state["question"]
    page_index = state["visual_page_index"]

    result = answer_visual_question_direct(
        question=question,
        page_index=page_index,
    )
    selected_asset = result["selected_asset"] or {}

    history = state.get("messages", [])
    updated_messages = [
        *history,
        {"role": "user", "content": question},
        {"role": "assistant", "content": result["answer"]},
    ]

    return {
        "answer": result["answer"],
        "insufficient_context": False,
        "retrieved_sources": [
            {
                "rank": 1,
                "route": "visual",
                "page_index": page_index,
                "kind": selected_asset.get("kind"),
                "region_id": selected_asset.get("region_id"),
                "reason_code": selected_asset.get("reason_code"),
                "asset_path": selected_asset.get("asset_path"),
                "page_render_path": selected_asset.get("page_render_path"),
            }
        ],
        "retrieved_context": (
            f"Visual route selected for page {page_index}. Asset={selected_asset.get('asset_path')}"
        ),
        "citations": [
            {
                "node_id": f"visual-page-{page_index}",
                "page_start": page_index,
                "page_end": page_index,
                "quote": result["citation_quote"],
            }
        ],
        "messages": updated_messages,
    }


# Reuse your existing retrieve_node and answer_node for the text path.
def build_routed_qa_graph(*, checkpointer: Any):
    builder = StateGraph(RoutedQAState)

    builder.add_node("route_question", route_question_node)
    builder.add_node("retrieve", retrieve_node)
    builder.add_node("answer", answer_node)
    builder.add_node("visual_answer", visual_answer_node)

    builder.add_edge(START, "route_question")
    builder.add_conditional_edges(
        "route_question",
        route_question_edge,
        {
            "retrieve": "retrieve",
            "visual_answer": "visual_answer",
        },
    )
    builder.add_edge("retrieve", "answer")
    builder.add_edge("answer", END)
    builder.add_edge("visual_answer", END)

    return builder.compile(checkpointer=checkpointer)


routed_checkpointer = InMemorySaver()
qa_graph_v2 = build_routed_qa_graph(checkpointer=routed_checkpointer)

print("Compiled routed QA graph")

Compiled routed QA graph


In [39]:
# Test that normal text questions still use the text path

question_4 = "What are the main sections in this document?"

result_4 = qa_graph_v2.invoke(
    {"question": question_4},
    config=config,
)

print(result_4["answer"])
print(json.dumps(result_4.get("citations", []), indent=2, default=str))


Provider List: https://docs.litellm.ai/docs/providers

The document covers two main mathematical and statistical topics: surds and statistics. The surds section discusses the simplest form of surds, like surds, and the rationalization of surds. The statistics section covers basic terms such as classes, class limits, frequency, class width (or class size/interval), class marks, and methods of classification (inclusive and exclusive), as well as the use of percentage bar diagrams.
[
  {
    "node_id": "b815aa3f3bffbb4281dcabc852830ce76d774fe5f12e8b5a16857c9a36a76e1e",
    "page_start": 35,
    "page_end": 35,
    "quote": "Surds"
  },
  {
    "node_id": "8e3b0ddeac29acb91fc8213300dff011907054423ed7f45e8615cf04398627eb",
    "page_start": 38,
    "page_end": 38,
    "quote": "Rationalization of surd"
  },
  {
    "node_id": "3f89c513feed4b6f4c18062db7a876f45104696d3b9a10c06eaf0fc484816a9d",
    "page_start": 119,
    "page_end": 119,
    "quote": "The information is shown in the percenta